# Limpieza de los Datos de la Tabla bronce.sucursales para Cargalos en la Capa Plata

Proposito del script:  
- Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
- Limpiar y estandarizar, columna por columna los datos.  
- Crear un archivo para los registros con inconsistencias con nombre "revision_productos_crediticios.parquet".
- Exportar la nueva tabla como un archivo con nombre "limpio_sucursales.parquet".

# Estableciendo Conexion

In [1]:
# Importando librerias y estableciendo conexion 
import pandas as pd 
from funciones import limpiar_texto, formato_requiere_garantia, formato_moneda
from conexiones_y_rutas import obtener_engine,obtener_ruta_archivo

engine = obtener_engine()
df_productos_crediticios = pd.read_sql(
    "SELECT * FROM bronce.productos_crediticios",
    con=engine
)
df_productos_tra = df_productos_crediticios.copy()

# Resumen Columnas

- **producto_id**: Identificador unicado de cada producto. 
- **nombre_producto**: Nombre del producto.     
- **tipo_credito**: Tipo de Credito (ejem: 'Personal', 'Vehicular').
- **tasa_nom_min**: Tasa nominal minima para dicho producto.
- **tasa_nom_max**: Tasa nominal maxima para dicho producto.
- **plazo_min_meses**: plazo mínimo de meses para dicho credito.
- **plazo_max_meses**: plazo maximo de meses para dicho credito.
- **monto_minimo**: Monto minimo que se tiene que aprobar para dicho credito.
- **monto_maximo**: Monto maximo que se tiene que aprobar para dicho credito.
- **requiere_garantia**: Valor booleano que verifica si el credito requiere o no garantia.
- **moneda**: Moneda en la que se realiza el credito (ejem: 'PEN' y'USD').

# Verificacion de la Calidad y Limpieza de los Datos

In [2]:
df_productos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   producto_id        8 non-null      int64  
 1   nombre_producto    8 non-null      object 
 2   tipo_credito       8 non-null      object 
 3   tasa_nom_min       8 non-null      float64
 4   tasa_nom_max       8 non-null      float64
 5   plazo_min_meses    8 non-null      int64  
 6   plazo_max_meses    8 non-null      int64  
 7   monto_minimo       8 non-null      float64
 8   monto_maximo       7 non-null      float64
 9   requiere_garantia  8 non-null      object 
 10  moneda             8 non-null      object 
dtypes: float64(4), int64(3), object(4)
memory usage: 832.0+ bytes


In [3]:
df_productos_tra.head()

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda
0,1,crédito personal libre disponibilidad,Personal,18.0,36.0,6,60,1000.0,50000.0,False,S/.
1,2,Crédito Hipotecario Vivienda,Hipotecario,7.5,11.0,60,360,50000.0,1000000.0,True,PEN
2,3,Crédito Vehicular,VehIcuLar,-2.5,18.0,12,72,10000.0,200000.0,yes,PEN
3,4,Crédito MYPE Capital de Trabajo,Microempresa,20.0,48.0,6,48,2000.0,100000.0,False,PEN
4,5,CrÉdItO De cOnSuMo,Consumo,24.0,42.0,3,36,500.0,20000.0,False,PEN


In [4]:
# Verifica si existen duplicados 
df_productos_tra[df_productos_tra.duplicated(keep=False)]

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda


In [5]:
df_productos_tra.drop_duplicates(inplace=True)
df_productos_tra.reset_index(drop=True,inplace=True)

In [6]:
df_productos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   producto_id        8 non-null      int64  
 1   nombre_producto    8 non-null      object 
 2   tipo_credito       8 non-null      object 
 3   tasa_nom_min       8 non-null      float64
 4   tasa_nom_max       8 non-null      float64
 5   plazo_min_meses    8 non-null      int64  
 6   plazo_max_meses    8 non-null      int64  
 7   monto_minimo       8 non-null      float64
 8   monto_maximo       7 non-null      float64
 9   requiere_garantia  8 non-null      object 
 10  moneda             8 non-null      object 
dtypes: float64(4), int64(3), object(4)
memory usage: 832.0+ bytes


## producto_id

In [7]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_productos_tra[df_productos_tra.producto_id <= 0]

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda


In [8]:
# Verifica si existen ids duplicados
# Resultados Esperados: Tabla Vacia
df_productos_tra.producto_id[df_productos_tra.producto_id.duplicated(keep=False)]

Series([], Name: producto_id, dtype: int64)

## nombre_producto

In [9]:
# Verifica si los nombres no tienen el formato adecuado
# Resultados Esperados: Tabla Vacia
df_productos_tra.nombre_producto[df_productos_tra.nombre_producto 
                                != df_productos_tra.nombre_producto.str.strip().str.title()]

0     crédito personal libre disponibilidad
3           Crédito MYPE Capital de Trabajo
4                       CrÉdItO De cOnSuMo 
5                         CRÉDITO EDUCATIVO
6                   crédito hipotecario usd
Name: nombre_producto, dtype: object

In [10]:
# Aplica el formato de texto adecuado
# Resultados Esperados: Crédito Mype Capital de Trabajo, Crédito de Consumo
df_productos_tra["nombre_producto"] = df_productos_tra.nombre_producto.apply(limpiar_texto)
df_productos_tra.nombre_producto[df_productos_tra.nombre_producto 
                                != df_productos_tra.nombre_producto.str.strip().str.title()]

3    Crédito Mype Capital de Trabajo
4                 Crédito de Consumo
Name: nombre_producto, dtype: object

In [11]:
df_productos_tra.nombre_producto.unique()

array(['Crédito Personal Libre Disponibilidad',
       'Crédito Hipotecario Vivienda', 'Crédito Vehicular',
       'Crédito Mype Capital de Trabajo', 'Crédito de Consumo',
       'Crédito Educativo', 'Crédito Hipotecario Usd',
       'Crédito Agropecuario'], dtype=object)

**Nota**: Voy arreglar manualmente algunos textos, que se que son incorrectos, porque, solo son 2 registros.

In [12]:
df_productos_tra.loc[3,"nombre_producto"] ="Crédito MYPE Capital De Trabajo"
df_productos_tra.loc[6,"nombre_producto"] ="Crédito Hipotecario USD"

## tipo_credito

In [13]:
# Resultados Esperados: 'Personal', 'Hipotecario', 'Vehicular', 'Microempresa', 'Consumo', 'n/a'
df_productos_tra.tipo_credito.unique()

array(['Personal', 'Hipotecario', ' VehIcuLar', 'Microempresa', 'Consumo',
       'personal', 'HIPOTECARIO', 'microempresa '], dtype=object)

In [14]:
# Aplica el formato de texto adecuado
# Resultados Esperados: 'Personal', 'Hipotecario', 'Vehicular', 'Microempresa', 'Consumo', 'n/a'
df_productos_tra["tipo_credito"] = df_productos_tra.tipo_credito.apply(limpiar_texto)
df_productos_tra.tipo_credito.unique()

array(['Personal', 'Hipotecario', 'Vehicular', 'Microempresa', 'Consumo'],
      dtype=object)

## tasa_nom_min

In [15]:
# Verifica si existen tasas nominales menores o iguales a 0
# Resultados Esperados: Tabla Vacia 
df_productos_tra.tasa_nom_min[df_productos_tra.tasa_nom_min <= 0 ]

2   -2.5
Name: tasa_nom_min, dtype: float64

In [16]:
# Valor absoluto para eliminar negativos 
# Resultados Esperados: Tabla Vacia 
df_productos_tra["tasa_nom_min"] = df_productos_tra.tasa_nom_min.apply(abs)
df_productos_tra.tasa_nom_min[df_productos_tra.tasa_nom_min <= 0 ]

Series([], Name: tasa_nom_min, dtype: float64)

## tasa_nom_max

**Nota**: Revisando los valores, para Crédito Hipotecario USD, se asigna un tasa maxima de 250, este valor es evidentemente atipico, pero, puede ser porque, es un tipo de credito especial o por un error al digitar, no veo sentido, arriesgarme y colocar una valor arbitrario, asi que lo voy a dejar como esta, pero lo voy a separar porque, necesita observacion.

In [17]:
# Resultados Esperados: Tabla Vacia 
df_productos_tra.tasa_nom_max[df_productos_tra.tasa_nom_max <= 0 ]

Series([], Name: tasa_nom_max, dtype: float64)

In [18]:
# Verifica si existen registros donde la tasa nominal maxima sea menor o igual a la tasa nominal minima 
# Resultadso Esperados: Tabla Vacias
df_productos_tra[["tasa_nom_min","tasa_nom_max"]][df_productos_tra.tasa_nom_max <= df_productos_tra.tasa_nom_min]

,tasa_nom_min,tasa_nom_max


In [19]:
df_revisar_tasa_max = df_productos_tra[df_productos_tra.producto_id == 7].copy()
df_revisar_tasa_max["ERROR"] = "TASA NOMINAL MAXIMA ATIPICA"

## plazo_min_meses

In [20]:
# Verifica si existen plazo de meses minimos menores o iguales a 0
# Resultados Esperados: Tabla Vacia 
df_productos_tra.plazo_min_meses[df_productos_tra.plazo_min_meses <= 0 ]

Series([], Name: plazo_min_meses, dtype: int64)

## plazo_max_meses

In [21]:
# Verifica si existen plazo de meses maximos menores o iguales a 0
# Resultados Esperados: Tabla Vacia 
df_productos_tra.plazo_max_meses[df_productos_tra.plazo_max_meses <= 0 ]

Series([], Name: plazo_max_meses, dtype: int64)

In [22]:
# Verifica si existen registros donde el plazo maximo sea menor o igual al plazo minimo de meses 
# Resultados Esperados: Tabla Vacia 
df_productos_tra[["plazo_min_meses","plazo_max_meses"]][df_productos_tra.plazo_max_meses 
                                                        <= df_productos_tra.plazo_min_meses]

,plazo_min_meses,plazo_max_meses


## monto_minimo

In [23]:
# Verifica si existen montos minimos menores o iguales a 0 
# Resultados Esperados: Tabla Vacia 
df_productos_tra.monto_minimo[df_productos_tra.monto_minimo <= 0]

Series([], Name: monto_minimo, dtype: float64)

## monto_maximo

**Nota**: Revisando los valores, para Crédito Educativo, no existe un monto maximo asignado, esto puede ser porque no tiene limite maximo, por error en la migracion de datos o muchas otras razones, no veo sentido, arriesgarme y colocar una valor arbitrario, asi que lo voy a dejar como esta, pero lo voy a separar porque necesita observacion.

In [24]:
# Verifica si existen montos maximos menores o iguales a 0 
# Resultados Esperados: Tabla Vacia 
df_productos_tra.monto_maximo[df_productos_tra.monto_maximo <= 0]

Series([], Name: monto_maximo, dtype: float64)

In [25]:
# Verifica si existen registros donde el monto maximo sea menor o igual al monto minimo 
# Resultados Esperados: Tabla Vacia 
df_productos_tra[["monto_minimo","monto_maximo"]][df_productos_tra.monto_maximo 
                                                        <= df_productos_tra.monto_minimo]

,monto_minimo,monto_maximo


In [26]:
df_revisar_mont_max = df_productos_tra[df_productos_tra.producto_id == 6].copy()
df_revisar_mont_max["ERROR"] = "NO SE REGISTRA MONTO MAXIMO"

## requiere_garantia

In [27]:
# Verifica si existen registros incorrectos
# Resultados Esperados: 'False', 'True', 'n/a'
df_productos_tra.requiere_garantia.unique()

array(['False', 'True', 'yes', 'No', 'false'], dtype=object)

In [28]:
# Resultados Esperados: Tabla Vacia
df_productos_tra["requiere_garantia"] = df_productos_tra.requiere_garantia.apply(formato_requiere_garantia)
# Transforma el tipo de dato de la columna a tipo boolean (para que acepte nulos tambien)
df_productos_tra["requiere_garantia"] = df_productos_tra.requiere_garantia.astype("boolean")
df_productos_tra.requiere_garantia.unique()

<BooleanArray>
[False, True]
Length: 2, dtype: boolean

## moneda

In [29]:
# Resultados Esperados: 'PEN', 'USD', 'n/a'
df_productos_tra.moneda.unique()

array(['S/.', 'PEN', 'PEN ', 'Sol', 'USD'], dtype=object)

In [30]:
# Resultados Esperados: Tabla Vacia
df_productos_tra["moneda"] = df_productos_tra.moneda.apply(formato_moneda)
df_productos_tra.moneda.unique()

array(['PEN', 'USD'], dtype=object)

# Limpiando Duplicados Luego de Limpieza

In [31]:
# Verifica si existen registros duplicados
# Resultados Esperados: Tabla Vacias
df_productos_tra[df_productos_tra.duplicated(keep=False)]

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda


In [32]:
# Elimina registros duplicados luego de limpieza
df_productos_tra.drop_duplicates(inplace=True)
df_productos_tra.reset_index(drop=True,inplace=True)

## producto_id V2

In [33]:
# Verifica si existen ids duplicados
# Resultados Esperados: Tabla Vacias
df_productos_tra[df_productos_tra.producto_id.duplicated(keep=False)]

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda


# Exportando la Tabla Limpia

In [34]:
df_productos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   producto_id        8 non-null      int64  
 1   nombre_producto    8 non-null      object 
 2   tipo_credito       8 non-null      object 
 3   tasa_nom_min       8 non-null      float64
 4   tasa_nom_max       8 non-null      float64
 5   plazo_min_meses    8 non-null      int64  
 6   plazo_max_meses    8 non-null      int64  
 7   monto_minimo       8 non-null      float64
 8   monto_maximo       7 non-null      float64
 9   requiere_garantia  8 non-null      boolean
 10  moneda             8 non-null      object 
dtypes: boolean(1), float64(4), int64(3), object(3)
memory usage: 784.0+ bytes


In [35]:
df_productos_tra.head()

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda
0,1,Crédito Personal Libre Disponibilidad,Personal,18.0,36.0,6,60,1000.0,50000.0,False,PEN
1,2,Crédito Hipotecario Vivienda,Hipotecario,7.5,11.0,60,360,50000.0,1000000.0,True,PEN
2,3,Crédito Vehicular,Vehicular,2.5,18.0,12,72,10000.0,200000.0,True,PEN
3,4,Crédito MYPE Capital De Trabajo,Microempresa,20.0,48.0,6,48,2000.0,100000.0,False,PEN
4,5,Crédito de Consumo,Consumo,24.0,42.0,3,36,500.0,20000.0,False,PEN


In [36]:
df_productos_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_productos_crediticios.parquet"),
    index=False
)

# Exportando los Registros de Productos Crediticos a Revisar

In [37]:
df_para_verificar = pd.concat([df_revisar_tasa_max,df_revisar_mont_max])
df_para_verificar.to_parquet(
    obtener_ruta_archivo("archivos_para_revision","revision_productos_crediticios.parquet"),
    index=False
)